# RegNet — Designing Network Design Spaces

**Paper**: Radosavovic, I., Kosaraju, R. P., Girshick, R., He, K., & Dollár, P. (2020).  
**"Designing Network Design Spaces"**  
CVPR 2020 · arXiv:2003.13678  
Facebook AI Research

---

## The Core Idea: Design Spaces, Not Design Points

Most neural architecture search (NAS) methods produce a single optimal architecture. RegNet takes a different approach: instead of finding one best network, it **designs a design space** — a constrained family of networks where all members are good.

Starting from the general **AnyNet** space (arbitrary per-stage widths, depths, group widths), the authors sample thousands of random networks, train each for a few epochs, and analyse which structural properties separate good networks from bad ones.  

They discover three key patterns that hold across all good AnyNet models:
1. **Widths grow monotonically** across stages.
2. **Widths follow a geometric (power-law) progression**: $w_j \approx w_0 \cdot \alpha^j$.
3. **Group width is constant** across all stages.

Enforcing these constraints produces the **RegNet** design space — a 6-parameter family. The best RegNets match or exceed NAS-found architectures at every compute budget.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('dark_background')
ACCENT = '#7EB8F7'
RED    = '#F77E7E'
GREEN  = '#7EF7A4'
GOLD   = '#F7C97E'

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('RegNet: Principled Network Design Space', fontsize=14, color='white', y=1.02)

# --- Panel 1: AnyNet → RegNet restriction ---
ax = axes[0]
ax.set_xlim(0, 6)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Design Space Refinement', color=ACCENT)

boxes = [
    (3, 9.0, 'AnyNet\n(arbitrary width/depth/group)', '#444', 'white'),
    (3, 7.2, 'Constraint: monotone widths', GOLD, 'black'),
    (3, 5.4, 'Constraint: geometric width', GOLD, 'black'),
    (3, 3.6, 'Constraint: shared group width', GOLD, 'black'),
    (3, 1.8, 'RegNet\n(6 parameters, all good)', GREEN, 'black'),
]
for x, y, label, col, tcol in boxes:
    ax.text(x, y, label, ha='center', va='center', fontsize=9, color=tcol,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=col, alpha=0.9))
ys = [b[1] for b in boxes]
for y0, y1 in zip(ys[:-1], ys[1:]):
    ax.annotate('', xy=(3, y1+0.45), xytext=(3, y0-0.45),
                arrowprops=dict(arrowstyle='->', color='#888'))

# --- Panel 2: Width progression (log-linear) ---
ax = axes[1]
stages = np.arange(4)
# RegNetX_032 widths: [96, 192, 432, 1008]
widths_032 = [96, 192, 432, 1008]
# RegNetX_160 widths: [256, 512, 896, 2048]
widths_160 = [256, 512, 896, 2048]
ax.semilogy(stages, widths_032, 'o-', color=ACCENT, lw=2, label='RegNetX-3.2GF', markersize=8)
ax.semilogy(stages, widths_160, 's-', color=GOLD, lw=2, label='RegNetX-16GF', markersize=8)
# Show geometric fit
w0, alpha = 96, 2.25
geo = [w0 * alpha**j for j in stages]
ax.semilogy(stages, geo, '--', color='#888', lw=1.5, label=f'Geometric: w₀={w0:.0f}, α={alpha:.2f}')
ax.set_xticks(stages)
ax.set_xticklabels([f'Stage {i}' for i in stages])
ax.set_ylabel('Width (log scale)', color='white')
ax.set_title('Width Progression', color=ACCENT)
ax.legend(fontsize=8)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_color('#444')

# --- Panel 3: ResBottleneckBlock diagram ---
ax = axes[2]
ax.set_xlim(0, 4)
ax.set_ylim(0, 11)
ax.axis('off')
ax.set_title('ResBottleneckBlock (RegNetX/Y)', color=ACCENT)

block_steps = [
    (2, 10.2, 'Input x', '#555'),
    (2, 8.9,  'Conv 1×1  BN  ReLU', ACCENT),
    (2, 7.6,  'GroupConv 3×3 (g=w/G)  BN  ReLU', ACCENT),
    (2, 6.3,  '[SE: avgpool→FC→FC]  (RegNetY)', GREEN),
    (2, 5.0,  'Conv 1×1  BN', ACCENT),
    (2, 3.7,  '+ residual (projected if needed)', GOLD),
    (2, 2.4,  'ReLU → output', '#555'),
]
for x, y, label, col in block_steps:
    ax.text(x, y, label, ha='center', va='center', fontsize=9,
            color='black' if col != '#555' else 'white',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=col, alpha=0.85))
bys = [b[1] for b in block_steps]
for y0, y1 in zip(bys[:-1], bys[1:]):
    ax.annotate('', xy=(2, y1+0.35), xytext=(2, y0-0.35),
                arrowprops=dict(arrowstyle='->', color='#888'))

plt.tight_layout()
plt.show()

## Mathematical Formulation

### The AnyNet Design Space

An AnyNet is parameterised per-stage: $(d_i, w_i, g_i, b_i)$ for $i=1..4$ (depth, width, group_width, bottleneck_ratio).  
This gives $\sim 10^{18}$ possible networks — far too large to search exhaustively.

### RegNet Quantisation

The authors find that good widths follow a log-linear (geometric) progression.  
Given initial width $w_0$, slope $w_a$, and quantisation parameter $w_m$:

$$w_j = w_0 + w_a \cdot j \quad \text{(continuous)}$$

Quantised (rounded to nearest multiple of 8):
$$\hat{w}_j = 8 \cdot \left\lfloor \frac{w_j}{8} + 0.5 \right\rfloor$$

### FLOPs approximation for one stage

For a stage with depth $d$, width $w$, group width $g$, input resolution $r$:
$$\text{FLOPs} \approx d \cdot r^2 \cdot \left( \frac{w^2}{g} \cdot k^2 + 2w^2 \right)$$

The $k^2 w^2/g$ term is the grouped conv; the $2w^2$ term covers the two 1×1 convs.  
Total network FLOPs = $\sum_i \text{FLOPs}_i$ (accounting for halved resolution per stage).

In [ ]:
"""Code walkthrough: RegNet building blocks."""
import sys
sys.path.insert(0, '../../../src')

import torch

# Show how group width controls the number of groups
def compute_groups(width: int, group_width: int) -> int:
    """Number of groups in the 3×3 grouped conv."""
    return max(1, width // group_width)

print('Width vs group_width → num_groups (for RegNetX_032):')
widths = [96, 192, 432, 1008]
gw = 48
for w in widths:
    g = compute_groups(w, gw)
    print(f'  width={w:4d}, group_width={gw}, groups={g:2d}')

print()
print('Note: group_width is fixed across ALL stages in RegNet.')
print('This is one of the key regularities the design space analysis found.')

In [ ]:
"""RegNet backbone usage example."""
import torch
from hale_core.nn.vision import build_vision_backbone, list_vision_backbones

# List all RegNet variants
regnet_backbones = sorted([n for n in list_vision_backbones() if 'regnet' in n])
print(f'Registered RegNet variants ({len(regnet_backbones)} total):')
for n in regnet_backbones:
    print(f'  {n}')

# Build and run RegNetX-3.2GF
model_x = build_vision_backbone('regnetx_032', img_size=224)
model_x.eval()

# Build and run RegNetY-3.2GF (same structure + SE)
model_y = build_vision_backbone('regnety_032', img_size=224)
model_y.eval()

x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out_x = model_x(x)
    out_y = model_y(x)

print(f'\nRegNetX-3.2GF:')
print(f'  pooled_output: {out_x.pooled_output.shape}')
print(f'  params: {model_x.num_parameters()/1e6:.1f}M')
for i, fm in enumerate(out_x.feature_maps):
    print(f'  stage {i}: {tuple(fm.shape)}')

print(f'\nRegNetY-3.2GF (+ SE):')
print(f'  pooled_output: {out_y.pooled_output.shape}')
print(f'  params: {model_y.num_parameters()/1e6:.1f}M  (+SE overhead)')

In [ ]:
"""Visualise RegNet family: params vs compute budget."""
import matplotlib.pyplot as plt
import torch
from hale_core.nn.vision import build_vision_backbone

plt.style.use('dark_background')
ACCENT = '#7EB8F7'
GOLD   = '#F7C97E'

# Approximate GFlop values from model names (suffix × 0.01)
suffixes   = ['002', '004', '008', '016', '032', '064']
gflops     = [0.2, 0.4, 0.8, 1.6, 3.2, 6.4]

params_x, params_y = [], []
for sfx in suffixes:
    mx = build_vision_backbone(f'regnetx_{sfx}', img_size=224)
    my = build_vision_backbone(f'regnety_{sfx}', img_size=224)
    params_x.append(mx.num_parameters() / 1e6)
    params_y.append(my.num_parameters() / 1e6)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gflops, params_x, 'o-', color=ACCENT, lw=2, label='RegNetX (no SE)', markersize=7)
ax.plot(gflops, params_y, 's-', color=GOLD,   lw=2, label='RegNetY (+ SE)',   markersize=7)
ax.set_xlabel('Approximate GFlops', color='white')
ax.set_ylabel('Parameters (M)', color='white')
ax.set_title('RegNet Family: Parameters vs Compute', color=ACCENT)
ax.legend()
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_color('#444')
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#0d0d1a')
plt.tight_layout()
plt.show()

## References

1. **Radosavovic, I., Kosaraju, R. P., Girshick, R., He, K., & Dollár, P. (2020).**  
   Designing Network Design Spaces.  
   *CVPR 2020.* arXiv:2003.13678

2. **Radosavovic, I., Johnson, J., Xie, S., Lo, W.-Y., & Dollár, P. (2019).**  
   On Network Design Spaces for Visual Recognition.  
   *ICCV 2019.* arXiv:1905.13214

3. **Hu, J., Shen, L., & Sun, G. (2018).**  
   Squeeze-and-Excitation Networks.  
   *CVPR 2018.* arXiv:1709.01507

4. **He, K., Zhang, X., Ren, S., & Sun, J. (2016).**  
   Deep Residual Learning for Image Recognition.  
   *CVPR 2016.* arXiv:1512.03385

5. **Xie, S., Girshick, R., Dollár, P., Tu, Z., & He, K. (2017).**  
   Aggregated Residual Transformations for Deep Neural Networks (ResNeXt).  
   *CVPR 2017.* arXiv:1611.05431